# Training results

This notebook reads cached training histories and split metrics. It never trains, evaluates a checkpoint, or changes methodology. A validation fallback keeps the notebook previewable before locked TEST artifacts arrive; final reporting uses TEST automatically when all four files exist.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
import sys

ROOT = Path.cwd()
if not (ROOT / 'reports').exists(): ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'scripts'))
from generate_report import _test_gate_diagnostics
MODELS = ('custom_cnn', 'resnet50', 'convnext_tiny', 'vit_b16')
test_paths = [ROOT / 'reports' / 'results' / model / 'test' / name for model in MODELS for name in ('metrics.json', 'predictions.csv', 'embeddings.npz')]
test_files_present = any(path.exists() for path in test_paths)
gate_diagnostics = _test_gate_diagnostics(ROOT)
if test_files_present and gate_diagnostics: raise RuntimeError('TEST artifacts are present but the lock/hashed-marker gate is invalid: ' + '; '.join(gate_diagnostics))
SPLIT = 'test' if test_files_present else 'val'
print('FINAL TEST RESULTS' if SPLIT == 'test' else 'VALIDATION PREVIEW ONLY - final TEST results require a valid lock and hashed completion markers.')

FINAL TEST RESULTS


In [2]:
from material_classifier.plotting import plot_training_curves

histories = {}
for model in MODELS:
    path = ROOT / 'artifacts' / 'checkpoints' / model / 'history.csv'
    if path.exists(): histories[model] = path
figure_paths = plot_training_curves(histories, ROOT / 'reports' / 'figures')
print(f'Loaded {len(histories)} histories and refreshed {len(figure_paths)} cached curve figures.')
combined = ROOT / 'reports' / 'figures' / 'training' / 'combined_val_macro_f1.png'
if combined.exists(): display(Image(filename=str(combined)))

Loaded 4 histories and refreshed 19 cached curve figures.


In [3]:
records = []
for model in MODELS:
    path = ROOT / 'reports' / 'results' / model / SPLIT / 'metrics.json'
    if not path.exists(): continue
    payload = json.loads(path.read_text(encoding='utf-8'))
    records.append({'model': model, 'split': SPLIT, 'accuracy': payload.get('accuracy'), 'balanced_accuracy': payload.get('balanced_accuracy'), 'macro_f1': payload.get('f1_macro'), 'weighted_f1': payload.get('f1_weighted'), 'roc_auc_macro': payload.get('roc_auc_macro'), 'log_loss': payload.get('log_loss'), 'ece': payload.get('ece'), 'best_epoch': payload.get('best_epoch')})
metrics = pd.DataFrame(records).set_index('model') if records else pd.DataFrame()
display(metrics.style.format(precision=3) if not metrics.empty else 'No cached metrics found.')

,split,accuracy,balanced_accuracy,macro_f1,weighted_f1,roc_auc_macro,log_loss,ece,best_epoch
model,,,,,,,,,
custom_cnn,test,0.498,0.498,0.490,0.490,0.889,1.177,0.099,3
resnet50,test,0.698,0.698,0.683,0.683,0.916,0.990,0.097,7
convnext_tiny,test,0.612,0.612,0.569,0.569,0.944,1.058,0.207,15
vit_b16,test,0.600,0.600,0.562,0.562,0.919,1.100,0.172,19
